# This file reads off unique_scrandle_cases_FULL and makes a new list, where only the unique PICTURES are listed.

In [1]:
import pandas as pd

# --- Paths (relative like you wanted) ---
input_path = "../data/grand_scraper_folder/unique_scrandle_cases_FULL.csv"
output_path = "../data/grand_scraper_folder/unique_scrandle_pictures_FULL.csv"


# --- Load data ---
df = pd.read_csv(input_path)

In [2]:
# --- Helper ---
def parse_occurrences(occ_str):
    if pd.isna(occ_str) or occ_str == "":
        return []

    parts = [p.strip() for p in occ_str.split("|")]
    return [(p.split(":")[0], p) for p in parts]


# --- Process ---
grouped = []

for image_hash, group in df.groupby("image_hash"):
    all_occ = []
    total_weight = 0
    weighted_rating_sum = 0

    for _, row in group.iterrows():
        occ_list = parse_occurrences(row["occurrences"])
        n = len(occ_list)

        total_weight += n
        weighted_rating_sum += row["rating"] * n
        all_occ.extend(occ_list)

    # --- Sort occurrences chronologically ---
    all_occ_sorted = sorted(all_occ, key=lambda x: x[0])
    # --- Rebuild occurrences string ---
    occurrences_str = " | ".join([x[1] for x in all_occ_sorted])

    # --- Weighted rating ---
    if total_weight > 0:
        rating = weighted_rating_sum / total_weight
    else:
        rating = group.iloc[0]["rating"]

    # --- Keep metadata (from first row) ---
    base = group.iloc[0].copy()
    base["rating"] = rating
    base["occurrences"] = occurrences_str

    grouped.append(base)


# --- Create dataframe ---
new_df = pd.DataFrame(grouped)

# --- Save ---
new_df.to_csv(output_path, index=False)

print("✅ Done")


✅ Done
